# Лабораторная работа
### Веб-приложение для демонстрации моделей машинного обучения
Нажмите **Run** (▶) на ячейке ниже — всё запустится автоматически.

In [ ]:
# ============================================================
# Лабораторная работа
# Веб-приложение для демонстрации моделей машинного обучения
# ============================================================

# Шаг 1: Установка библиотек
import subprocess
subprocess.run(["pip", "install", "gradio", "scikit-learn", "matplotlib", "numpy", "-q"])

# Шаг 2: Импорт
import gradio as gr
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

print("Библиотеки загружены!")

# Шаг 3: Функции

def generate_dataset(n_samples, noise):
    X, y = make_classification(
        n_samples=n_samples,
        n_features=2,
        n_informative=2,
        n_redundant=0,
        n_clusters_per_class=1,
        flip_y=noise / 100.0,
        random_state=42
    )
    return X, y


def plot_decision_boundary(model, X, y, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    h = 0.05
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    axes[0].contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    scatter = axes[0].scatter(X[:, 0], X[:, 1], c=y,
                               cmap='RdBu', edgecolors='k',
                               s=40, linewidth=0.5)
    axes[0].set_title('Граница решений: ' + title, fontsize=12)
    axes[0].set_xlabel('Признак 1')
    axes[0].set_ylabel('Признак 2')
    plt.colorbar(scatter, ax=axes[0])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42)
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['Класс 0', 'Класс 1'])
    disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
    axes[1].set_title('Матрица ошибок (тест)', fontsize=12)

    plt.tight_layout()
    return fig


def train_and_evaluate(model_name, n_samples, noise,
                        dt_max_depth, knn_n_neighbors, lr_c):
    X, y = generate_dataset(int(n_samples), noise)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42)

    if model_name == 'Дерево решений':
        model = DecisionTreeClassifier(
            max_depth=int(dt_max_depth), random_state=42)
    elif model_name == 'Метод k ближайших соседей':
        model = KNeighborsClassifier(n_neighbors=int(knn_n_neighbors))
    elif model_name == 'Логистическая регрессия':
        model = LogisticRegression(
            C=float(lr_c), max_iter=500, random_state=42)
    else:
        return 'Выберите модель!', None

    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc  = accuracy_score(y_test,  model.predict(X_test))

    result_text = (
        'Модель: ' + model_name + '\n' +
        'Обучающих примеров: ' + str(len(X_train)) +
        '  |  Тестовых: ' + str(len(X_test)) + '\n' +
        'Точность на обучении: ' + str(round(train_acc, 4)) +
        ' (' + str(round(train_acc * 100, 2)) + '%)\n' +
        'Точность на тесте:    ' + str(round(test_acc, 4)) +
        ' (' + str(round(test_acc * 100, 2)) + '%)'
    )

    fig = plot_decision_boundary(model, X, y, model_name)
    return result_text, fig


def update_description(model_name):
    descriptions = {
        'Дерево решений':
            'Делит пространство признаков на прямоугольные области. '
            'max_depth ограничивает глубину: малая — недообучение, большая — переобучение.',
        'Метод k ближайших соседей':
            'Классифицирует точку по большинству голосов k ближайших соседей. '
            'Малое k — сложная граница, большое k — сглаженная.',
        'Логистическая регрессия':
            'Строит линейную границу между классами. '
            'C — обратная сила регуляризации: большое C = меньше штрафа.'
    }
    return descriptions.get(model_name, '')


# Шаг 4: Интерфейс и запуск

with gr.Blocks(title='ML Demo', theme=gr.themes.Soft()) as demo:

    gr.Markdown('# Демонстрация моделей машинного обучения')
    gr.Markdown('Выберите модель, настройте параметры и нажмите **Обучить**.')

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### Датасет')
            n_samples = gr.Slider(100, 1000, value=300, step=50,
                                   label='Количество примеров')
            noise = gr.Slider(0, 40, value=10, step=1,
                               label='Уровень шума (%)')

            gr.Markdown('### Модель')
            model_choice = gr.Dropdown(
                choices=['Дерево решений',
                         'Метод k ближайших соседей',
                         'Логистическая регрессия'],
                value='Дерево решений',
                label='Выберите модель'
            )
            model_desc = gr.Textbox(
                value=update_description('Дерево решений'),
                label='Описание',
                lines=3,
                interactive=False
            )

            gr.Markdown('### Гиперпараметры')
            dt_depth = gr.Slider(1, 15, value=3, step=1,
                                  label='[Дерево] max_depth')
            knn_k = gr.Slider(1, 20, value=5, step=1,
                               label='[KNN] n_neighbors')
            lr_c = gr.Slider(0.01, 10.0, value=1.0, step=0.1,
                              label='[Лог. рег.] C')

            btn = gr.Button('Обучить модель', variant='primary')

        with gr.Column(scale=2):
            gr.Markdown('### Результаты')
            result_text = gr.Textbox(label='Метрики качества',
                                      lines=5, interactive=False)
            result_plot = gr.Plot(label='Графики')

    model_choice.change(fn=update_description,
                         inputs=model_choice,
                         outputs=model_desc)

    btn.click(
        fn=train_and_evaluate,
        inputs=[model_choice, n_samples, noise,
                dt_depth, knn_k, lr_c],
        outputs=[result_text, result_plot]
    )

demo.launch(share=True, debug=False)
